In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(
    r"C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation"
)

RETRIEVAL_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "knowledge"
    / "knowledge_base_retrieval.csv"
)

OUTPUT_DIR = (
    BASE_DIR
    / "outputs"
    / "embeddings"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df = pd.read_csv(RETRIEVAL_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

Shape: (15979, 6)
Columns: ['document_id', 'prompt', 'context', 'response', 'source_dataset', 'retrieval_text']


,document_id,prompt,context,response,source_dataset,retrieval_text
0,0,What is (are) Monkeypox Virus Infections ?,NaN,Monkeypox is a rare viral disease. It occurs m...,MedQuAD,Question: What is (are) Monkeypox Virus Infect...
1,1,Do you have information about Vitamin K,NaN,Summary : Vitamins are substances that your bo...,MedQuAD,Question: Do you have information about Vitami...
2,2,What is (are) phosphoribosylpyrophosphate synt...,NaN,Phosphoribosylpyrophosphate synthetase superac...,MedQuAD,Question: What is (are) phosphoribosylpyrophos...
3,3,What are the symptoms of Kallmann syndrome 6 ?,NaN,What are the signs and symptoms of Kallmann sy...,MedQuAD,Question: What are the symptoms of Kallmann sy...
4,4,Is Marfan syndrome inherited ?,NaN,How is Marfan syndrome inherited? Marfan syndr...,MedQuAD,Question: Is Marfan syndrome inherited ?\n\nAn...


In [2]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = (
    "pritamdeka/"
    "BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
)

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Embedding model loaded.")

Loading embedding model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\raich\anaconda3\envs\major_env\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\raich\.cache\huggingface\hub\models--pritamdeka--BioBERT-mnli-snli-scinli-scitail-mednli-stsb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.47k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/669k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [3]:
test_embedding = embedding_model.encode(
    ["This is a test medical sentence."],
    convert_to_numpy=True
)

print(
    "Embedding shape:",
    test_embedding.shape
)

Embedding shape: (1, 768)


In [4]:
texts = (
    df["retrieval_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

print(
    "Number of documents:",
    len(texts)
)

Number of documents: 15979


In [5]:
embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings generated.")
print("Shape:", embeddings.shape)
print("Data type:", embeddings.dtype)

Batches:   0%|          | 0/500 [00:00<?, ?it/s]

Embeddings generated.
Shape: (15979, 768)
Data type: float32


In [6]:
print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", embeddings.shape[1])

print(
    "Any NaN:",
    np.isnan(embeddings).any()
)

print(
    "Any infinite values:",
    np.isinf(embeddings).any()
)
norms = np.linalg.norm(
    embeddings,
    axis=1
)

print(
    "Minimum norm:",
    norms.min()
)

print(
    "Maximum norm:",
    norms.max()
)

print(
    "Mean norm:",
    norms.mean()
)

Number of embeddings: 15979
Embedding dimension: 768
Any NaN: False
Any infinite values: False
Minimum norm: 0.9999998
Maximum norm: 1.0000001
Mean norm: 1.0


In [7]:
EMBEDDINGS_PATH = (
    OUTPUT_DIR
    / "knowledge_base_embeddings.npy"
)

np.save(
    EMBEDDINGS_PATH,
    embeddings
)

print(
    "Saved:",
    EMBEDDINGS_PATH
)

Saved: C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation\outputs\embeddings\knowledge_base_embeddings.npy


In [8]:
metadata = df[
    [
        "document_id",
        "source_dataset",
        "prompt"
    ]
].copy()

METADATA_PATH = (
    OUTPUT_DIR
    / "embedding_metadata.csv"
)

metadata.to_csv(
    METADATA_PATH,
    index=False
)

print(
    "Saved:",
    METADATA_PATH
)

Saved: C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation\outputs\embeddings\embedding_metadata.csv


In [9]:
print(
    "Embeddings:",
    embeddings.shape[0]
)

print(
    "Metadata:",
    len(metadata)
)

print(
    "Document IDs:",
    metadata["document_id"].nunique()
)

assert embeddings.shape[0] == len(metadata)

assert metadata["document_id"].is_unique

print(
    "\nEmbedding ↔ metadata alignment verified."
)

Embeddings: 15979
Metadata: 15979
Document IDs: 15979

Embedding ↔ metadata alignment verified.


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

test_queries = [
    "What are the symptoms of monkeypox?",
    "What is Marfan syndrome?",
    "How is vitamin K deficiency treated?",
    "What causes diabetes?",
    "What are the symptoms of Kallmann syndrome?"
]

query_embeddings = embedding_model.encode(
    test_queries,
    convert_to_numpy=True,
    normalize_embeddings=True
)

similarities = cosine_similarity(
    query_embeddings,
    embeddings
)

print(
    "Similarity matrix shape:",
    similarities.shape
)

Similarity matrix shape: (5, 15979)


In [11]:
TOP_K = 5

for query_index, query in enumerate(test_queries):

    scores = similarities[query_index]

    top_indices = np.argsort(
        scores
    )[-TOP_K:][::-1]

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        print(
            f"\nRank {rank}"
        )

        print(
            "Score:",
            round(
                float(scores[idx]),
                4
            )
        )

        print(
            "Source:",
            df.iloc[idx]["source_dataset"]
        )

        print(
            "Prompt:",
            df.iloc[idx]["prompt"]
        )

        print(
            "Response:",
            str(
                df.iloc[idx]["response"]
            )[:500]
        )


QUERY: What are the symptoms of monkeypox?

Rank 1
Score: 0.7022
Source: MedQuAD
Prompt: What is (are) Monkeypox Virus Infections ?
Response: Monkeypox is a rare viral disease. It occurs mostly in central and western Africa. Wild rodents and squirrels carry it, but it is called monkeypox because scientists saw it first in lab monkeys. In 2003, it was reported in prairie dogs and humans in the U.S.     Centers for Disease Control and Prevention

Rank 2
Score: 0.4737
Source: MedQuAD
Prompt: What are the symptoms of Moyamoya disease ?
Response: What are the signs and symptoms of Moyamoya disease? The Human Phenotype Ontology provides the following list of signs and symptoms for Moyamoya disease. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate number of patients (when available) Abnormality 